# Jyotish LoRA: DPO step 2, sample the model's own answers

Samples several answers per prompt from the SFT model (adapter on, temperature sampling). Scoring happens on the laptop.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes
!pip uninstall -y -q torchao

In [ ]:
import contextlib, glob, json, re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ---------- settings ----------
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
QLORA = False
ADAPTER_HINT = ""             # substring of the adapter path to pick when several inputs carry adapters
N_SAMPLES = 3
GREEDY = False               # True: one greedy answer per prompt (as the evaluation does) and N_SAMPLES is ignored
TEMPERATURE, TOP_P = 0.8, 0.95
BATCH = 6                     # prompts per generate call (x N_SAMPLES sequences)
MAX_NEW = 600
LIMIT = 0                     # >0: only the first LIMIT prompts (smoke test)
OUT = "/kaggle/working/samples.jsonl"

def find(name):
    hits = glob.glob("/kaggle/input/**/" + name, recursive=True)
    assert hits, name + " not found"
    return hits[0]

adapters = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
adapters = [a for a in adapters if ADAPTER_HINT in a] or adapters
ADAPTER = os.path.dirname(adapters[-1])
prompts = [json.loads(l) for l in open(find("prompts.jsonl"), encoding="utf-8") if l.strip()]
if LIMIT:
    prompts = prompts[:LIMIT]
print("adapter:", ADAPTER, "| prompts:", len(prompts))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if QLORA:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
else:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

In [ ]:
done = set()
if os.path.exists(OUT):
    done = {r["id"] for r in map(json.loads, open(OUT, encoding="utf-8"))}
todo = sorted([p for p in prompts if p["id"] not in done], key=lambda p: len(str(p["messages"])))
clean = lambda s: re.sub(r"<think>.*?</think>", "", s, flags=re.S).strip()
t0 = time.time()
torch.manual_seed(0)
for i in range(0, len(todo), BATCH):
    batch = todo[i:i + BATCH]
    texts = [tokenizer.apply_chat_template(b["messages"], tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False) for b in batch]
    enc = tokenizer(texts, return_tensors="pt", padding=True).to("cuda")
    n_ret = 1 if GREEDY else N_SAMPLES
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, repetition_penalty=1.05, pad_token_id=tokenizer.pad_token_id,
                             num_return_sequences=n_ret,
                             **({"do_sample": False} if GREEDY else {"do_sample": True, "temperature": TEMPERATURE, "top_p": TOP_P}))
    n = enc["input_ids"].shape[1]
    with open(OUT, "a", encoding="utf-8") as f:
        for j, b in enumerate(batch):
            answers = [clean(tokenizer.decode(o[n:], skip_special_tokens=True)) for o in out[j * n_ret:(j + 1) * n_ret]]
            f.write(json.dumps({"id": b["id"], "answers": answers}, ensure_ascii=False) + "\n")
    torch.cuda.empty_cache()
    if (i // BATCH) % 10 == 0:
        print("%d/%d prompts, %.1f min" % (min(i + BATCH, len(todo)), len(todo), (time.time() - t0) / 60), flush=True)
print("ALL DONE in %.2f h" % ((time.time() - t0) / 3600))